# 06 · Dünya'ya Etki (S5) ve Gecikme Analizi (S2 ön çalışma)
**Tez:** Güneş Döngüleri 23–25 Boyunca Güneş Patlamaları, Güneş Lekeleri ve Jeomanyetik Etkiler
**Hazırlayan:** Tunahan İnci · **Defter sürümü:** 1.0 (05.11.2026)

**Bölümler**
0. **Nihai flare listesi (S4):** Referans kalibrasyon senaryosuyla tüm akılar GOES-16 ölçeğine getirilir. Doymuş 4 patlamaya Hudson ve ark. (2023) yeniden hesaplanmış tepe değerleri ek sütun olarak eklenir. Döngü tablosunun kesin değerleri.
1. **Üst üste bindirilmiş olay analizi (superposed epoch):** Patlama anını sıfır kabul edip −2 ile +5 gün arasında Dst ve Kp'nin ortalama davranışı. Sınıflara göre (M1–M5, M5–X1, ≥X1). Karşılaştırma için aynı dönemlerden rastgele seçilmiş zamanlar.
2. **Konum etkisi:** Güneş diskinin doğu kenarı, merkezi ve batı kenarındaki X sınıfı patlamaların etkisi farklı mı?
3. **Fırtına olasılığı:** Bir X patlamasından sonraki 12–96 saatte fırtına çıkma olasılığı, rastgele zamanlara göre kaç kat?
4. **Yıllık fırtınalar ve gecikme:** Fırtına günleri leke sayısını kaç ay geriden izliyor? Aynı analiz M+X patlamaları için de yapılır.
5. **Vaka çalışmaları:** Ekim–Kasım 2003 (Cadılar Bayramı) ve Mayıs 2024 fırtınaları.

**Bilinen sınır:** Jeomanyetik fırtınaları asıl oluşturan, patlamaya eşlik eden koronal kütle atımlarıdır (CME). CME kataloğu henüz eklenmedi; bu analiz patlamayı CME için bir gösterge olarak kullanır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.dates as mdates
KOK = Path('/content/drive/MyDrive/gunes_tezi')
ISLENMIS, SEKIL = KOK/'data'/'processed', KOK/'figures'
RENK = {'M1–M5': '#3fa0c9', 'M5–X1': '#c27a00', '≥X1': '#c4432c'}
print('Hazır')

In [ ]:
# Saf fonksiyonlar (test edilebilir)
import numpy as np, pandas as pd

def kumelenmeyi_ayikla(zaman, aki, pencere_saat=48):
    # Büyükten küçüğe giderek, kabul edilmiş bir olaya ±pencere içinde olanları atar.
    # Böylece aynı aktif dönemdeki ardışık patlamalar tek olay sayılır.
    sira = np.argsort(-np.asarray(aki))
    t = pd.to_datetime(pd.Series(zaman)).values.astype('datetime64[s]').astype(np.int64)
    kabul, kabul_t = [], []
    pen = pencere_saat * 3600
    for i in sira:
        if all(abs(t[i] - k) > pen for k in kabul_t):
            kabul.append(i); kabul_t.append(t[i])
    return np.sort(np.array(kabul, dtype=int))

def olay_matrisi(seri, zamanlar, once_saat=48, sonra_saat=120):
    # Saatlik seriden, her olay zamanının etrafındaki pencereyi satır olarak çıkarır.
    s = seri.sort_index()
    t0 = s.index[0]
    deger = s.values.astype(float)
    konum = ((pd.to_datetime(pd.Series(zamanlar)).dt.floor('h') - t0) / pd.Timedelta(hours=1)).values.astype(int)
    gecikme = np.arange(-once_saat, sonra_saat + 1)
    idx = konum[:, None] + gecikme[None, :]
    gecerli = (idx >= 0) & (idx < len(deger))
    M = np.full(idx.shape, np.nan)
    M[gecerli] = deger[idx[gecerli]]
    return gecikme, M

def rastgele_esle(zamanlar, alt, ust, gun=180, tekrar=1, tohum=0):
    # Her olay için ±gun içinde rastgele bir zaman seçer (güneş döngüsü evresini korur).
    rng = np.random.default_rng(tohum)
    z = pd.to_datetime(pd.Series(zamanlar)).values
    out = []
    for _ in range(tekrar):
        kayma = rng.uniform(-gun, gun, len(z)) * 86400
        r = z + (kayma * 1e9).astype('timedelta64[ns]')
        r = np.clip(r, np.datetime64(alt), np.datetime64(ust))
        out.append(pd.DatetimeIndex(r))
    return out

def firtina_var_mi(dst_saatlik, zamanlar, esik=-50, bas_saat=12, son_saat=96):
    gec, M = olay_matrisi(dst_saatlik, zamanlar, 0, son_saat)
    pencere = M[:, bas_saat:]
    return np.nanmin(pencere, axis=1) <= esik

## 0 · Nihai flare listesi (S4 referans senaryosu)
GOES-10'un GOES-16 ölçeğindeki katsayısı = GOES-14 katsayısı / 1,091 (sabit fark düzeltilmiş dakika köprüsü). Eski grubun diğer uyduları 03 defterindeki iç oranlarla ölçeklenir.

In [ ]:
ncei = pd.read_parquet(ISLENMIS/'flare_ncei.parquet')
ana = pd.read_csv(ISLENMIS/'03_kalibrasyon_katsayilari.csv').set_index('uydu')
ESKI = [8, 9, 10, 11, 12]
a10 = ana.faktor[14] / 1.091
faktor = ana.faktor.to_dict()
for u in ESKI:
    if u in faktor: faktor[u] = faktor[u] * a10
print('S4 katsayıları (GOES-16 = 1):', {k: round(v, 3) for k, v in sorted(faktor.items())})

fl = ncei.copy()
fl['faktor'] = fl.uydu.map(faktor).fillna(1.0)
fl['aki'] = fl.pik_aki / fl.faktor
fl['sinif'] = pd.cut(np.log10(fl.aki), [-np.inf, -7, -6, -5, -4, np.inf], labels=list('ABCMX'), right=False).astype(str)
TABAN = {'A': 1e-8, 'B': 1e-7, 'C': 1e-6, 'M': 1e-5, 'X': 1e-4}
fl['sinif_metin'] = [f"{'≥' if d else ''}{h}{a / TABAN[h]:.1f}" for h, a, d in zip(fl.sinif, fl.aki, fl.doymus)]

# Hudson ve ark. (2023) Tablo 7, yeni ölçek tepe akıları (W/m²) — doymuş olaylar için ek bilgi
HUDSON = {'2003-11-04': 4.32e-3, '2001-04-02': 2.96e-3, '2003-10-28': 2.57e-3, '2005-09-07': 2.46e-3}
fl['hudson2023_aki'] = np.nan
for g, v in HUDSON.items():
    sec = fl.doymus & (fl.pik.dt.strftime('%Y-%m-%d') == g)
    fl.loc[sec, 'hudson2023_aki'] = v

# Konum: HEK'ten en yakın kayıt (±10 dk); (0,0) bilinmiyor demek
hek = pd.read_parquet(ISLENMIS/'flare_hek_temiz.parquet')[['event_peaktime', 'hpc_x', 'hpc_y', 'ar_noaanum']]
hek['event_peaktime'] = hek.event_peaktime.astype('datetime64[ns]'); hek = hek.sort_values('event_peaktime')
fl['pik'] = fl.pik.astype('datetime64[ns]')
fl = pd.merge_asof(fl.sort_values('pik'), hek, left_on='pik', right_on='event_peaktime',
                   tolerance=pd.Timedelta(minutes=10), direction='nearest').drop(columns='event_peaktime')
bilinmez = fl.hpc_x.isna() | ((fl.hpc_x == 0) & (fl.hpc_y == 0))
fl['bolge'] = np.select([bilinmez, fl.hpc_x < -500, fl.hpc_x > 500], ['bilinmiyor', 'doğu kenarı', 'batı kenarı'], 'merkez')

fl.to_parquet(ISLENMIS/'flare_final.parquet', index=False)
print(f'Kaydedildi: flare_final.parquet ({len(fl):,} patlama)')
print('Sınıf dağılımı:', fl.sinif.value_counts().reindex(list('ABCMX')).fillna(0).astype(int).to_dict())
print('X sınıfı konum dağılımı:', fl[fl.sinif == 'X'].bolge.value_counts().to_dict())

sn = pd.read_parquet(ISLENMIS/'sn_aylik_1749.parquet').set_index('tarih').sn
DONGU = {'23': ('2000-01-01', '2008-11-30'), '24': ('2008-12-01', '2019-11-30'), '25': ('2019-12-01', '2025-12-31')}
satir = []
for d, (b, s) in DONGU.items():
    sec = (fl.pik >= b) & (fl.pik <= s)
    t_sn = sn[b:s].sum()
    mx = (sec & fl.sinif.isin(['M', 'X'])).sum(); x = (sec & (fl.sinif == 'X')).sum()
    satir.append([d, mx, x, round(t_sn), round(mx / t_sn * 1000, 1), round(x / t_sn * 1000, 2)])
dongu_final = pd.DataFrame(satir, columns=['Döngü', 'M+X', 'X', 'Σ SN', 'M+X /1000SN', 'X /1000SN'])
dongu_final.to_csv(ISLENMIS/'06_dongu_final_S4.csv', index=False)
r = dongu_final.set_index('Döngü')['M+X /1000SN']
print(f"\nD25/D23 = {r['25']/r['23']:.3f} | D25/D24 = {r['25']/r['24']:.3f} | D23/D24 = {r['23']/r['24']:.3f}")
dongu_final

## 1 · Üst üste bindirilmiş olay analizi
Patlamalar 48 saat içinde birbirine yakınsa yalnızca en büyüğü tutulur (kümelenme ayıklaması). Her sınıf için Dst'nin **medyan** eğrisi ve çeyrekler arası bandı çizilir. Gri bant: her olay için ±180 gün içinden seçilen rastgele zamanların medyan eğrisinin %95 aralığı (200 tekrar). Patlama eğrisi bu bandın altına iniyorsa etki rastlantı değildir.

In [ ]:
omni = pd.read_parquet(ISLENMIS/'omni2_saatlik.parquet').set_index('zaman').sort_index()
omni = omni[~omni.index.duplicated()].asfreq('h')
DST, KP = omni.dst, omni.kp
ALT, UST = '2000-01-03', '2025-12-25'

SINIF = {'M1–M5': (1e-5, 5e-5), 'M5–X1': (5e-5, 1e-4), '≥X1': (1e-4, 1)}
olaylar = {}
for ad, (a, b) in SINIF.items():
    s = fl[(fl.aki >= a) & (fl.aki < b) & (fl.pik >= ALT) & (fl.pik <= UST)]
    k = kumelenmeyi_ayikla(s.pik, s.aki, 48)
    olaylar[ad] = s.iloc[k]
    print(f'{ad}: {len(s):5d} patlama → kümelenme ayıklandıktan sonra {len(k):4d} bağımsız olay')

def sea_ciz(seri, etiket, dosya, tekrar=200):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True, constrained_layout=True)
    ozet = {}
    for i, (ad, o) in enumerate(olaylar.items()):
        g, M = olay_matrisi(seri, o.pik)
        med = np.nanmedian(M, 0); q1, q3 = np.nanpercentile(M, [25, 75], 0)
        rastgele = [np.nanmedian(olay_matrisi(seri, r)[1], 0) for r in rastgele_esle(o.pik, ALT, UST, 180, tekrar, i)]
        rl, ru = np.nanpercentile(rastgele, [2.5, 97.5], 0)
        gun = g / 24
        ax[i].fill_between(gun, rl, ru, color='grey', alpha=.25, label='Rastgele %95')
        ax[i].fill_between(gun, q1, q3, color=RENK[ad], alpha=.15)
        ax[i].plot(gun, med, color=RENK[ad], lw=2, label=f'{ad} medyan (n={len(o)})')
        ax[i].axvline(0, color='k', lw=.8); ax[i].grid(alpha=.25); ax[i].set_xlabel('Patlamadan sonra gün')
        ax[i].legend(loc='lower left', fontsize=8); ax[i].set_title(ad)
        sonra = (g >= 12) & (g <= 96)
        en = np.nanargmin(med[sonra]) if 'Dst' in etiket else np.nanargmax(med[sonra])
        ozet[ad] = {'n': len(o), 'tepe_saat': int(g[sonra][en]), 'tepe_deger': round(float(med[sonra][en]), 1),
                    'once_ort': round(float(np.nanmean(med[g < 0])), 1),
                    'rastgele_ust_sinir': round(float((rl if 'Dst' in etiket else ru)[sonra][en]), 1)}
    ax[0].set_ylabel(etiket)
    fig.suptitle(f'Büyük patlamalardan sonra {etiket.split(" ")[0]}: üst üste bindirilmiş olay analizi')
    fig.savefig(SEKIL/dosya, dpi=200); plt.show()
    return pd.DataFrame(ozet).T

sea_dst = sea_ciz(DST, 'Dst (nT)', '06_sea_dst.png')
print(sea_dst.to_string())
sea_kp = sea_ciz(KP, 'Kp', '06_sea_kp.png')
print(sea_kp.to_string())

## 2 · Konum etkisi: X sınıfı patlamalar
Güneş'in Dünya'ya bakan yüzünün merkezinden ve batı yarısından çıkan patlamaların CME'lerinin Dünya'ya ulaşma olasılığı daha yüksektir. Burada X sınıfı (kümelenme ayıklanmış) patlamalar konumlarına göre ayrılır. Konum HEK kaydından gelir (helyoprojektif x, yay saniyesi; Güneş yarıçapı ≈ 960″).

In [ ]:
xo = olaylar['≥X1']
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
konum_ozet = {}
for bolge, renk in [('doğu kenarı', '#3451c4'), ('merkez', '#c4432c'), ('batı kenarı', '#c27a00')]:
    o = xo[xo.bolge == bolge]
    if len(o) < 3: continue
    g, M = olay_matrisi(DST, o.pik)
    med = np.nanmedian(M, 0)
    ax.plot(g / 24, med, lw=2, color=renk, label=f'{bolge} (n={len(o)})')
    sonra = (g >= 12) & (g <= 96)
    konum_ozet[bolge] = {'n': len(o), 'en_dusuk_medyan_dst': round(float(np.nanmin(med[sonra])), 1)}
ax.axvline(0, color='k', lw=.8); ax.grid(alpha=.25); ax.legend()
ax.set_xlabel('Patlamadan sonra gün'); ax.set_ylabel('Medyan Dst (nT)')
ax.set_title('X sınıfı patlamalar: konuma göre Dst tepkisi')
fig.savefig(SEKIL/'06_konum_dst.png', dpi=200); plt.show()
print(pd.DataFrame(konum_ozet).T.to_string())

## 3 · Fırtına olasılığı
Bir patlamadan sonraki **12–96 saat** içinde Dst'nin −50 nT (orta) veya −100 nT (şiddetli) altına inme olasılığı. Karşılaştırma: aynı dönemlerden rastgele seçilmiş zamanlar (her olay için 20 rastgele zaman). **Kat** = patlama sonrası olasılık / rastgele olasılık.

In [ ]:
satir = []
gruplar = {**{k: v for k, v in olaylar.items()},
           **{f'≥X1 · {b}': olaylar['≥X1'][olaylar['≥X1'].bolge == b] for b in ['doğu kenarı', 'merkez', 'batı kenarı']}}
for ad, o in gruplar.items():
    if len(o) < 5: continue
    for esik in (-50, -100):
        p = firtina_var_mi(DST, o.pik, esik).mean()
        rz = rastgele_esle(o.pik, ALT, UST, 180, 20, 7)
        pr = np.mean([firtina_var_mi(DST, r, esik).mean() for r in rz])
        satir.append([ad, len(o), f'Dst ≤ {esik}', round(p * 100, 1), round(pr * 100, 1), round(p / pr, 2) if pr else np.nan])
olasilik = pd.DataFrame(satir, columns=['Grup', 'n', 'Fırtına', 'Patlama sonrası %', 'Rastgele %', 'Kat'])
olasilik.to_csv(ISLENMIS/'06_firtina_olasiligi.csv', index=False)
olasilik

## 4 · Yıllık fırtınalar ve gecikme
Her yıl için fırtına günü sayıları (Dst ≤ −50, Dst ≤ −100, Kp ≥ 5). Ardından aylık seriler 13 ay düzleştirilip leke sayısıyla **çapraz korelasyon** hesaplanır: pozitif gecikme, serinin leke sayısını kaç ay geriden izlediğini gösterir.

Not: Düzleştirilmiş seriler arasındaki korelasyonun anlamlılığı otokorelasyon nedeniyle şişer. Burada yalnızca **gecikmenin yerine** bakıyoruz; anlamlılık testleri S2 defterinde blok bootstrap ile yapılacak.

In [ ]:
gunluk = pd.read_parquet(ISLENMIS/'omni2_gunluk.parquet').set_index('zaman')
yillik = pd.DataFrame({
    'Dst≤−50 gün': (gunluk.dst_min <= -50).groupby(gunluk.index.year).sum(),
    'Dst≤−100 gün': (gunluk.dst_min <= -100).groupby(gunluk.index.year).sum(),
    'Kp≥5 gün': (gunluk.kp_max >= 5).groupby(gunluk.index.year).sum(),
    'M+X': fl[fl.sinif.isin(['M', 'X'])].groupby(fl.pik.dt.year).size(),
    'SN': sn['2000':'2025'].groupby(sn['2000':'2025'].index.year).mean().round(1),
}).fillna(0)
yillik.index.name = 'yıl'
yillik.to_csv(ISLENMIS/'06_yillik.csv')
print(yillik.to_string())

satir = []
for d, (b, s) in DONGU.items():
    g_ = gunluk[b:s]; t_sn = sn[b:s].sum()
    satir.append([d, int((g_.dst_min <= -50).sum()), int((g_.dst_min <= -100).sum()),
                  round((g_.dst_min <= -50).sum() / t_sn * 1000, 1), round((g_.dst_min <= -100).sum() / t_sn * 1000, 2)])
dongu_firtina = pd.DataFrame(satir, columns=['Döngü', 'Dst≤−50 gün', 'Dst≤−100 gün', '≤−50 /1000SN', '≤−100 /1000SN'])
dongu_firtina.to_csv(ISLENMIS/'06_dongu_firtina.csv', index=False)
print('\nDöngü bazında fırtına günleri:'); print(dongu_firtina.to_string(index=False))

aylik = pd.DataFrame({
    'SN': sn['2000':'2025'],
    'M+X': fl[fl.sinif.isin(['M', 'X'])].set_index('pik').resample('MS').size(),
    'Dst≤−50 gün': (gunluk.dst_min <= -50).resample('MS').sum(),
    'Dst≤−100 gün': (gunluk.dst_min <= -100).resample('MS').sum(),
}).fillna(0)['2000':'2025']
duz = aylik.rolling(13, center=True).mean().dropna()
gecikmeler = range(-48, 49)
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
gecikme_ozet = {}
for c, renk in [('M+X', '#3fa0c9'), ('Dst≤−50 gün', '#3451c4'), ('Dst≤−100 gün', '#c4432c')]:
    r = [duz.SN.corr(duz[c].shift(-L)) for L in gecikmeler]
    ax.plot(list(gecikmeler), r, lw=2, color=renk, label=c)
    en = int(np.nanargmax(r)); gecikme_ozet[c] = {'en_iyi_gecikme_ay': list(gecikmeler)[en], 'r': round(r[en], 3), 'r_gecikme0': round(r[48], 3)}
ax.axvline(0, color='k', lw=.8); ax.grid(alpha=.25); ax.legend()
ax.set_xlabel('Gecikme (ay) — pozitif: seri leke sayısının arkasından geliyor'); ax.set_ylabel('Korelasyon (13 ay düzleştirilmiş)')
ax.set_title('Leke sayısı ile patlama ve fırtına aktivitesi arasındaki gecikme')
fig.savefig(SEKIL/'06_gecikme.png', dpi=200); plt.show()
print(pd.DataFrame(gecikme_ozet).T.to_string())

fig, ax = plt.subplots(figsize=(13, 4.5), constrained_layout=True)
ax2 = ax.twinx()
ax.bar(yillik.index, yillik['Dst≤−100 gün'], color='#c4432c', alpha=.7, label='Şiddetli fırtına günü (Dst ≤ −100)')
ax.bar(yillik.index, yillik['Dst≤−50 gün'] - yillik['Dst≤−100 gün'], bottom=yillik['Dst≤−100 gün'], color='#3451c4', alpha=.45, label='Orta fırtına günü (−100 < Dst ≤ −50)')
ax2.plot(yillik.index, yillik.SN, color='#b86e00', lw=2.5, label='Yıllık ortalama leke sayısı')
ax.set_ylabel('Fırtına günü / yıl'); ax2.set_ylabel('Leke sayısı'); ax.grid(alpha=.25)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc='upper right', fontsize=8)
ax.set_title('Yıllık jeomanyetik fırtına günleri ve güneş lekesi sayısı, 2000–2025')
fig.savefig(SEKIL/'06_yillik_firtina.png', dpi=200); plt.show()

## 5 · Vaka çalışmaları
Üst panel: patlamalar (dikey çizgi yüksekliği = pik akı, log ölçek). Orta: saatlik Dst. Alt: Kp.

In [ ]:
def vaka(bas, son, baslik, dosya):
    f = fl[(fl.pik >= bas) & (fl.pik <= son) & (fl.aki >= 1e-6)]
    o = omni[bas:son]
    fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
    renk = f.sinif.map({'C': '#4cc38a', 'M': '#c27a00', 'X': '#c4432c'})
    ax[0].vlines(f.pik, 1e-6, f.aki, colors=renk, lw=1.5); ax[0].set_yscale('log'); ax[0].set_ylim(1e-6, 5e-3)
    for e, h in [(1e-5, 'M'), (1e-4, 'X')]:
        ax[0].axhline(e, color='grey', ls='--', lw=.7); ax[0].text(pd.Timestamp(bas), e, f' {h}', va='bottom', color='grey')
    for _, r in f.nlargest(3, 'aki').iterrows():
        ax[0].annotate(r.sinif_metin, (r.pik, r.aki), xytext=(3, 3), textcoords='offset points', fontsize=8)
    ax[0].set_ylabel('Pik akı (W/m²)')
    ax[1].plot(o.index, o.dst, color='#3451c4', lw=1.5); ax[1].axhline(-100, color='grey', ls='--', lw=.7); ax[1].set_ylabel('Dst (nT)')
    ax[2].bar(o.index, o.kp, width=1/24, color=np.where(o.kp >= 5, '#c4432c', '#8a94ad')); ax[2].set_ylabel('Kp'); ax[2].set_ylim(0, 9.5)
    for a in ax: a.grid(alpha=.25)
    ax[2].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    fig.suptitle(baslik); fig.savefig(SEKIL/dosya, dpi=200); plt.show()
    print(f'{baslik}: en düşük Dst = {o.dst.min():.0f} nT ({o.dst.idxmin():%Y-%m-%d %H:%M}), en yüksek Kp = {o.kp.max():.1f}, '
          f'M+X sayısı = {int(f.sinif.isin(["M", "X"]).sum())}, X sayısı = {int((f.sinif == "X").sum())}')

vaka('2003-10-25', '2003-11-08', 'Ekim–Kasım 2003 (Cadılar Bayramı fırtınaları)', '06_vaka_2003.png')
vaka('2024-05-05', '2024-05-16', 'Mayıs 2024 (Gannon fırtınası)', '06_vaka_2024.png')